# Bayyinah – Burnout Prediction: Data Consistency + Preprocessing

**What this notebook does**

- **Part A** – fixes logically impossible values in the dataset (it is *synthetic*, so some columns were generated independently and don't agree with each other).
- **Part B** – drops the columns we must not use, creates one shared train/test split and builds the preprocessor.

**What it does NOT do:** it does not train any model. The output files go to two teammates: one builds the regression model (`burnout_score`) and one builds the classification model (`burnout_level`).

**How to run:** Runtime → Run all. If the CSV is not in the Colab folder yet, the first cell will ask you to upload it.

## 0. Setup

In [ ]:
import os, json
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_PATH = "mental_health_burnout_tech_2026.csv"
CONSISTENT_PATH = "mental_health_burnout_tech_2026_consistent.csv"
CHANGES_PATH = "changes_log.csv"
OUT_DIR = "prepared"
RANDOM_STATE = 42

# In Colab: ask for the file if it isn't uploaded yet
if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Please upload", DATA_PATH)
    files.upload()

In [ ]:
raw = pd.read_csv(DATA_PATH)   # 'raw' is never modified, we always work on copies

print("Shape:", raw.shape)
print("Missing values:", int(raw.isna().sum().sum()))
print("Duplicate rows:", int(raw.duplicated().sum()))
raw.head(3)

---
# PART A – Data consistency

The dataset is synthetic. Columns like `age`, `years_experience`, `years_at_company` and `seniority_level` were generated separately, so we get impossible people (for example a 22-year-old with 15 years at the company).

We only fix what is **impossible**. We don't invent new relationships and we don't touch the targets. In this part we only change three columns: `years_experience`, `years_at_company`, `seniority_level`.

## A1. Gender filter

We keep Male, Female and Prefer not to say (team decision).

In [ ]:
KEEP_GENDERS = ["Male", "Female", "Prefer not to say"]

df = raw[raw["gender"].isin(KEEP_GENDERS)].copy()

print("Rows before:", len(raw))
print("Rows removed:", len(raw) - len(df))
print("Rows after :", len(df))

# snapshot BEFORE any fix, so we can compare before/after later (A5)
before = df.copy()

## A2. Detect the problems (before fixing)

Three rules:

1. `years_experience > age - 18` → nobody starts working before 18.
2. `years_at_company > years_experience` → you can't be at one company longer than your whole career.
3. `seniority_level` needs a minimum number of years: Junior 0, Mid 2, Senior 5, Lead 6, Manager 6, Principal 10.

*Where these numbers come from:* there is no scientific standard for this, every company sets its own levels. We based them on published engineering career ladders and on real job postings from tech companies such as Asana (Senior 5+ years, Tech Lead 6+ years, Senior Staff 10+ years). Lead was first set to 7 and changed to 6 to match the Tech Lead requirement. These are our assumptions and we state them in the report.

We count first so we know how big the problem is before changing anything.

In [ ]:
MIN_YEARS = {"Junior": 0, "Mid": 2, "Senior": 5, "Lead": 6, "Manager": 6, "Principal": 10}

def check_rules(d):
    """Returns one True/False mask per rule (True = the row breaks the rule)."""
    return {
        "Rule 1: experience > age - 18":       d["years_experience"] > d["age"] - 18,
        "Rule 2: years_at_company > experience": d["years_at_company"] > d["years_experience"],
        "Rule 3: seniority vs experience":     d["years_experience"] < d["seniority_level"].map(MIN_YEARS),
    }

def print_rule_counts(d, title):
    masks = check_rules(d)
    print(title)
    for name, m in masks.items():
        print(f"  {name:<40} {int(m.sum()):>7,}  ({m.mean()*100:.2f}%)")
    any_rule = pd.concat(masks, axis=1).any(axis=1)
    print(f"  {'Rows breaking at least one rule':<40} {int(any_rule.sum()):>7,}  ({any_rule.mean()*100:.2f}%)")
    return {k: int(v.sum()) for k, v in masks.items()}

counts_before = print_rule_counts(df, "BEFORE fixing:")

## A3. Fix (order matters)

The order is 1 → 2 → 3 because each rule depends on the one before it: Rule 2 uses the experience that Rule 1 already capped, and Rule 3 uses the final experience.

Every single change is written to a log (`employee_id`, column, old value, new value, rule) so anyone can trace exactly what we touched.

In [ ]:
change_parts = []

def log_changes(mask, column, new_values, rule):
    """Save old/new values for the rows in `mask`, then apply the change to df."""
    if mask.sum() == 0:
        return
    change_parts.append(pd.DataFrame({
        "employee_id": df.loc[mask, "employee_id"].values,
        "column": column,
        "old_value": df.loc[mask, column].astype(object).values,
        "new_value": pd.Series(new_values).astype(object).values,
        "rule": rule,
    }))
    df.loc[mask, column] = new_values

### Rule 1 – cap `years_experience` at `age - 18`

In [ ]:
max_exp = df["age"] - 18          # the most experience this person can possibly have

m1 = df["years_experience"] > max_exp
log_changes(m1, "years_experience", max_exp[m1].values, "Rule 1")
print("Rule 1 – rows fixed:", int(m1.sum()))

### Rule 2 – `years_at_company` can't be more than `years_experience`

Two cases:

- **(a) Raise experience.** If setting experience to `ceil(years_at_company)` still fits under `age - 18`, we raise the experience. The tenure value is believable, the experience was just too low.
- **(b) Cap tenure.** If that would break Rule 1 (the person is too young for that tenure), the tenure itself is impossible, so we cap `years_at_company` at `years_experience` (minimum 0.1).

**Small edge case:** in case (b), if experience is 0 the tenure becomes 0.1, and 0.1 is still bigger than 0. For those rows we apply case (a) one more time: experience goes to `ceil(0.1) = 1`. This always respects Rule 1 because the youngest employee is 22. These rows are logged as `Rule 2c` so they are easy to find.

In [ ]:
m2 = df["years_at_company"] > df["years_experience"]
needed_exp = np.ceil(df["years_at_company"]).astype(int)

# (a) raise experience when the person is old enough for it
m2a = m2 & (needed_exp <= max_exp)
log_changes(m2a, "years_experience", needed_exp[m2a].values, "Rule 2a (raise experience)")

# (b) otherwise cap the tenure at the experience (min 0.1)
m2b = m2 & ~m2a
new_tenure = df.loc[m2b, "years_experience"].clip(lower=0.1).astype(float).values
log_changes(m2b, "years_at_company", new_tenure, "Rule 2b (cap years_at_company)")

# (c) edge case: tenure 0.1 with experience 0 -> experience becomes 1
m2c = df["years_at_company"] > df["years_experience"]
new_exp_c = np.ceil(df.loc[m2c, "years_at_company"]).astype(int).values
assert (new_exp_c <= max_exp[m2c].values).all(), "edge-case fix would break Rule 1"
log_changes(m2c, "years_experience", new_exp_c, "Rule 2c (edge case: experience 0 -> 1)")

print("Rule 2a – experience raised :", int(m2a.sum()))
print("Rule 2b – tenure capped     :", int(m2b.sum()))
print("Rule 2c – edge case (0 -> 1):", int(m2c.sum()))

### Rule 3 – seniority vs experience (smallest change first)

If someone's title needs more years than they have, either the title is wrong or the experience number is wrong. We pick the **smaller change**:

- **(a) Raise experience.** If the person is old enough (`age - 18` ≥ the minimum years for their level), we raise `years_experience` to that minimum. The title stays. A wrong number is a smaller change than taking away someone's job title.
- **(b) Lower the title.** Only if the person is too young to have that experience, we lower `seniority_level` to the highest level their experience allows (Principal > Lead > Senior > Mid > Junior). A Manager who doesn't qualify goes to the highest **non-manager** level. We never promote anyone.

Raising experience in (a) can't break the earlier rules: it stays under `age - 18` by construction, and more experience can only make Rule 2 *more* true.

*Note: the original plan only lowered titles. We changed it because lowering about 16,400 titles distorted the data a lot more (see A5 and the Suggestions section).*

In [ ]:
LADDER = ["Principal", "Lead", "Senior", "Mid", "Junior"]   # highest first, Manager is not in the ladder

def highest_allowed_level(exp):
    for level in LADDER:
        if exp >= MIN_YEARS[level]:
            return level

level_min = df["seniority_level"].map(MIN_YEARS)
m3 = df["years_experience"] < level_min

# (a) old enough -> raise experience to the level's minimum, keep the title
m3a = m3 & (level_min <= max_exp)
log_changes(m3a, "years_experience", level_min[m3a].values, "Rule 3a (raise experience)")

# (b) too young -> lower the title
m3b = m3 & ~m3a
new_levels = df.loc[m3b, "years_experience"].map(highest_allowed_level).values
log_changes(m3b, "seniority_level", new_levels, "Rule 3b (lower seniority)")

print("Rule 3a – experience raised:", int(m3a.sum()))
print("Rule 3b – title lowered    :", int(m3b.sum()))

In [ ]:
changes_log = pd.concat(change_parts, ignore_index=True)
print("Total logged changes:", len(changes_log))
changes_log.groupby(["rule", "column"]).size().rename("changes").to_frame()

In [ ]:
changes_log.sample(8, random_state=1)

### Re-check: all counts must be 0 now

In [ ]:
counts_after = print_rule_counts(df, "AFTER fixing:")
assert all(v == 0 for v in counts_after.values()), "Some rows still break a rule!"
print("\nAll consistency rules pass.")

## A4. Report-only checks (nothing is changed here)

### Outliers (IQR method)

A value is counted as an outlier if it is below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`. We only **report** them. Extreme values (like 70 work hours) are possible in real life and can be exactly the cases that matter for burnout, so deleting them would hurt the model.

The 0/1 columns and the ID are skipped because IQR means nothing for them.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
binary_cols = [c for c in numeric_cols if df[c].nunique() <= 2]
iqr_cols = [c for c in numeric_cols if c not in binary_cols and c != "employee_id"]

rows = []
for c in iqr_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((df[c] < low) | (df[c] > high)).sum())
    rows.append({"column": c, "lower_bound": round(low, 2), "upper_bound": round(high, 2),
                 "min": df[c].min(), "max": df[c].max(),
                 "outliers": n, "outliers_%": round(n / len(df) * 100, 2)})

outlier_report = pd.DataFrame(rows).sort_values("outliers", ascending=False).reset_index(drop=True)
outlier_report

**Comment:** all values stay inside ranges that make sense (scores 1–10, sleep 3–10 hours, work hours 35–72). The "outliers" are just the tails of the distributions, not errors, so we keep them.

### Median salary by country and seniority

In [ ]:
LEVEL_ORDER = ["Junior", "Mid", "Senior", "Lead", "Manager", "Principal"]

salary_table = (df.pivot_table(index="country", columns="seniority_level",
                               values="salary_usd", aggfunc="median")
                  .reindex(columns=LEVEL_ORDER)
                  .sort_values("Senior", ascending=False)
                  .round(0).astype(int))
display(salary_table)

floor = df["salary_usd"].min()
print(f"\nShare of employees exactly at the minimum salary ({floor:,} USD):")
print((df["salary_usd"].eq(floor).groupby(df["country"]).mean() * 100).round(1).sort_values(ascending=False).to_string())

**Comment (is it plausible?)**

- **Mostly yes.** Salary goes up with seniority in every country (Junior < Mid < Senior < Lead < Manager < Principal), and the country order is realistic: USA is highest, then Canada / UK / Australia, then Western Europe and Singapore, with Brazil and India lowest.
- **One artifact:** the generator has a hard minimum of 40,000 USD. In India about 77% of employees sit exactly on that minimum (Brazil about 45%), so the median for Junior, Mid and Senior in India is the same number. In reality those salaries would be lower and different from each other.
- **Side effect of Rule 3:** salary was generated from the *original* seniority. For the employees whose title is lowered in Rule 3b, they keep their old salary, so medians of the lower levels move up a little. We accept this because Part A is only allowed to touch three columns.

We don't change anything here, we just mention it in the report as a limitation.

## A5. Did the fixes change the meaning of the data?

The fixes are only OK if the data still "says" the same thing as before. Here we measure that.

### Rows changed per rule

In [ ]:
n_rows = len(df)
per_rule = (changes_log.groupby("rule")["employee_id"].nunique()
            .rename("rows_changed").to_frame())
per_rule["%_of_rows"] = (per_rule["rows_changed"] / n_rows * 100).round(2)
display(per_rule)

rule2_rows = changes_log.loc[changes_log["rule"].str.startswith("Rule 2"), "employee_id"].nunique()
total_changed = changes_log["employee_id"].nunique()
print(f"Rule 2 (a+b+c together): {rule2_rows:,} rows ({rule2_rows / n_rows * 100:.2f}%)")
print(f"Rows with at least one change: {total_changed:,} of {n_rows:,} ({total_changed / n_rows * 100:.2f}%)")
print("(a row can be changed by more than one rule, so the rules don't add up to the total)")

### Only the three allowed columns changed

In [ ]:
CHANGED_COLS = ["years_experience", "years_at_company", "seniority_level"]

other_cols = [c for c in df.columns if c not in CHANGED_COLS]
assert before[other_cols].equals(df[other_cols]), "A column outside the allowed three was modified!"
assert before["burnout_score"].equals(df["burnout_score"]) and before["burnout_level"].equals(df["burnout_level"])
print("OK – every other column (including both targets) is identical to the original.")

### Distributions of the changed columns, before vs after

In [ ]:
for c in ["years_experience", "years_at_company"]:
    comp = pd.concat([before[c].describe(), df[c].describe()], axis=1, keys=["before", "after"]).round(3)
    comp["diff"] = (comp["after"] - comp["before"]).round(3)
    print(f"\n{c}")
    print(comp.to_string())

In [ ]:
sen = pd.concat([before["seniority_level"].value_counts(), df["seniority_level"].value_counts()],
                axis=1, keys=["before", "after"]).reindex(LEVEL_ORDER)
sen["before_%"] = (sen["before"] / n_rows * 100).round(2)
sen["after_%"] = (sen["after"] / n_rows * 100).round(2)
sen["diff_pp"] = (sen["after_%"] - sen["before_%"]).round(2)
sen

**Comment:** in the original file seniority was generated with no link to experience at all (a Principal had the same average experience as a Junior), so about one in five titles was impossible. Because Rule 3 raises experience first, `years_experience` goes up a bit (mean +0.5 years, and nobody has 0 years any more), and only the titles of employees who are too young for them are lowered, so `seniority_level` moves only a few percentage points. `years_at_company` barely moves.

### Correlation with `burnout_score`, before vs after

If a fix changed the relationship between a feature and the target, it would show up here. Differences should be about 0. Anything above 0.02 gets flagged.

In [ ]:
corr_features = [c for c in numeric_cols if c not in ("employee_id", "burnout_score")]

corr_table = pd.DataFrame({
    "before": before[corr_features].corrwith(before["burnout_score"]),
    "after": df[corr_features].corrwith(df["burnout_score"]),
})
corr_table["abs_diff"] = (corr_table["after"] - corr_table["before"]).abs()
corr_table["flag"] = np.where(corr_table["abs_diff"] > 0.02, "CHECK", "")
display(corr_table.round(4).sort_values("abs_diff", ascending=False))

n_flag = int((corr_table["abs_diff"] > 0.02).sum())
print(f"Biggest change: {corr_table['abs_diff'].max():.4f}  |  features flagged (> 0.02): {n_flag}")

`seniority_level` is text, so it is not in the correlation table. We check it another way: the average `burnout_score` per level before vs after.

In [ ]:
sen_burn = pd.concat([before.groupby("seniority_level")["burnout_score"].mean(),
                      df.groupby("seniority_level")["burnout_score"].mean()],
                     axis=1, keys=["mean_burnout_before", "mean_burnout_after"]).reindex(LEVEL_ORDER)
sen_burn["diff"] = sen_burn["mean_burnout_after"] - sen_burn["mean_burnout_before"]
display(sen_burn.round(3))

def explained_by_seniority(d, target):
    """Share of the target's variance explained by seniority_level alone (0 = nothing, 1 = everything)."""
    group_mean = d.groupby("seniority_level")[target].transform("mean")
    return 1 - ((d[target] - group_mean) ** 2).sum() / ((d[target] - d[target].mean()) ** 2).sum()

for target in ["burnout_score", "salary_usd"]:
    print(f"Variance of {target:<14} explained by seniority_level:  "
          f"before {explained_by_seniority(before, target):.3f}  ->  after {explained_by_seniority(df, target):.3f}")

**Comment (be honest about this in the report):** this is the one place where the fix is not completely neutral.

In the original data Lead and Manager have a much higher average burnout (about 7.4) than the other levels (about 4.8), so the generator used the job title to create burnout. When Rule 3b lowers a title, the employee takes their burnout with them, so Junior and Mid look a little more burned out than before and `seniority_level` explains a bit less of `burnout_score` (see the numbers above).

This is why Rule 3 raises experience first and lowers titles only for employees who are too young: lowering every impossible title (about 16,400 rows) would have dropped the explained variance from 0.166 to about 0.102. With the current rule it is 0.135. The remaining drop is the price of removing titles that are truly impossible.

### `burnout_level` distribution, before vs after

Must be identical because we never touch the targets and we don't drop rows in A3.

In [ ]:
lvl = pd.concat([before["burnout_level"].value_counts(normalize=True),
                 df["burnout_level"].value_counts(normalize=True)],
                axis=1, keys=["before_%", "after_%"]).mul(100).round(2)
lvl = lvl.reindex(["Low", "Moderate", "High", "Severe"])
lvl["diff"] = lvl["after_%"] - lvl["before_%"]
lvl

## A6. Save

The original file stays as it is. We write a new "consistent" file and the changes log.

In [ ]:
df.to_csv(CONSISTENT_PATH, index=False)
changes_log.to_csv(CHANGES_PATH, index=False)
print("Saved:", CONSISTENT_PATH, df.shape)
print("Saved:", CHANGES_PATH, changes_log.shape)

---
# PART B – Preprocessing and train/test split

From here we use the consistent data from Part A.

In [ ]:
data = pd.read_csv(CONSISTENT_PATH)
print(data.shape)

## B1. Drop columns

- `employee_id` is just a row number. A model could memorise it but it carries no meaning.
- The **leakage columns** are mental-health outcomes (stress, PHQ-9, GAD-7, ...). They are almost another way of measuring burnout, and a real HR system would never have them. If we kept them the model would look great here and be useless in Bayyinah.

In [ ]:
ID_COL = "employee_id"
LEAKAGE_COLS = ["stress_score", "phq9_score", "phq9_category", "gad7_score", "gad7_category",
                "seeks_mental_health_support", "job_change_intention"]
TARGET_REG = "burnout_score"
TARGET_CLF = "burnout_level"

data = data.drop(columns=[ID_COL] + LEAKAGE_COLS)
print("Columns left:", data.shape[1])

## B2. Targets

`burnout_level` is just `burnout_score` cut into ranges, so each target would give away the other one. Both are removed from X.

In [ ]:
y_reg = data[TARGET_REG]
y_clf = data[TARGET_CLF]
X = data.drop(columns=[TARGET_REG, TARGET_CLF])

print("X:", X.shape, "| y_reg:", y_reg.shape, "| y_clf:", y_clf.shape)

## B3. Two feature lists

- `FEATURES_FULL` – everything that is left.
- `FEATURES_HR` – the same without the personal/lifestyle columns, because a real HR system doesn't know how much you sleep or whether you go to therapy.

Later the teammates train with both lists to see how much accuracy we lose with the realistic one.

In [ ]:
PERSONAL_COLS = ["sleep_hours_per_night", "exercise_days_per_week", "therapy_access", "uses_therapy"]

FEATURES_FULL = X.columns.tolist()
FEATURES_HR = [c for c in FEATURES_FULL if c not in PERSONAL_COLS]

# column types (used by the preprocessor)
NUMERIC_COLS = X.select_dtypes(include="number").columns.tolist()
ORDINAL_COLS = ["company_size"]
ONEHOT_COLS = [c for c in FEATURES_FULL if c not in NUMERIC_COLS + ORDINAL_COLS]

COMPANY_SIZE_ORDER = ["Startup (1-50)", "Small (51-200)", "Mid (201-1000)",
                      "Large (1001-5000)", "Enterprise (5000+)"]
assert set(X["company_size"].unique()) == set(COMPANY_SIZE_ORDER)

print(f"FEATURES_FULL ({len(FEATURES_FULL)}):", FEATURES_FULL)
print(f"\nFEATURES_HR   ({len(FEATURES_HR)}):", FEATURES_HR)
print(f"\nNumeric ({len(NUMERIC_COLS)}):", NUMERIC_COLS)
print("Ordinal:", ORDINAL_COLS)
print("One-hot:", ONEHOT_COLS)

## B4. One shared train/test split

80% train / 20% test, `random_state=42`, stratified on `burnout_level` so the four classes have the same share in both parts.

We split the **row indices once** and use them for X and for both targets. This way the regression model and the classification model are tested on exactly the same employees and their results can be compared.

In [ ]:
idx_train, idx_test = train_test_split(
    X.index, test_size=0.20, random_state=RANDOM_STATE, stratify=y_clf
)

X_train, X_test = X.loc[idx_train], X.loc[idx_test]
y_train_reg, y_test_reg = y_reg.loc[idx_train], y_reg.loc[idx_test]
y_train_clf, y_test_clf = y_clf.loc[idx_train], y_clf.loc[idx_test]

assert len(set(idx_train) & set(idx_test)) == 0      # no employee is in both parts
print("Train:", X_train.shape, "| Test:", X_test.shape)

## B5. Preprocessor

- **Numeric** → median imputer + `StandardScaler`.
- **company_size** → `OrdinalEncoder` with the real order Startup < Small < Mid < Large < Enterprise.
- **seniority_level and the other categoricals** → most-frequent imputer + `OneHotEncoder(handle_unknown="ignore")`.

The data has no missing values now, but the real system may send some later, so the imputers are there to stop the pipeline from crashing. `company_size` also gets a most-frequent imputer before the encoder for the same reason (without it a missing value would come out as NaN).

It is fitted on the **training data only**. If the scaler saw the test data, information from the test set would leak into training and the test score would be too optimistic.

In [ ]:
def build_preprocessor(feature_list):
    """Builds the (unfitted) preprocessor for the given list of feature names."""
    num = [c for c in feature_list if c in NUMERIC_COLS]
    ordi = [c for c in feature_list if c in ORDINAL_COLS]
    cat = [c for c in feature_list if c in ONEHOT_COLS]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[COMPANY_SIZE_ORDER])),
    ])
    onehot_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num),
            ("company_size", ordinal_pipe, ordi),
            ("cat", onehot_pipe, cat),
        ],
        remainder="drop",   # any column that is not in feature_list is ignored
    )

preprocessor_full = build_preprocessor(FEATURES_FULL).fit(X_train)   # train only
preprocessor_hr = build_preprocessor(FEATURES_HR).fit(X_train)       # train only

print("FULL -> train", preprocessor_full.transform(X_train).shape, "| test", preprocessor_full.transform(X_test).shape)
print("HR   -> train", preprocessor_hr.transform(X_train).shape, "| test", preprocessor_hr.transform(X_test).shape)

Quick test: a row with missing values and a job role the model never saw should still go through.

In [ ]:
dirty = X_test.iloc[[0]].copy()
dirty["salary_usd"] = np.nan
dirty["company_size"] = np.nan
dirty["work_mode"] = np.nan
dirty["job_role"] = "Some New Role"

out = preprocessor_full.transform(dirty)
print("Output shape:", out.shape, "| NaN in output:", bool(np.isnan(out).any()))

## B6. Save everything to `prepared/`

`X_train.csv` and `X_test.csv` are saved **raw (not scaled)** and contain all FULL columns. The teammates put the preprocessor and the model in one sklearn `Pipeline`, so scaling happens inside the pipeline. For the HR version they use the same X files with `preprocessor_hr`, which simply ignores the personal columns.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

X_train.to_csv(f"{OUT_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{OUT_DIR}/X_test.csv", index=False)
y_train_reg.to_csv(f"{OUT_DIR}/y_train_reg.csv", index=False)
y_test_reg.to_csv(f"{OUT_DIR}/y_test_reg.csv", index=False)
y_train_clf.to_csv(f"{OUT_DIR}/y_train_clf.csv", index=False)
y_test_clf.to_csv(f"{OUT_DIR}/y_test_clf.csv", index=False)

joblib.dump(preprocessor_full, f"{OUT_DIR}/preprocessor_full.pkl")
joblib.dump(preprocessor_hr, f"{OUT_DIR}/preprocessor_hr.pkl")

feature_lists = {
    "FEATURES_FULL": FEATURES_FULL,
    "FEATURES_HR": FEATURES_HR,
    "numeric": NUMERIC_COLS,
    "categorical_ordinal": ORDINAL_COLS,
    "categorical_onehot": ONEHOT_COLS,
    "company_size_order": COMPANY_SIZE_ORDER,
    "personal_columns_removed_in_HR": PERSONAL_COLS,
    "targets": {"regression": TARGET_REG, "classification": TARGET_CLF},
    "burnout_level_classes": ["Low", "Moderate", "High", "Severe"],
    "split": {"test_size": 0.20, "random_state": RANDOM_STATE, "stratify": TARGET_CLF},
}
with open(f"{OUT_DIR}/feature_lists.json", "w") as f:
    json.dump(feature_lists, f, indent=2)

for name in sorted(os.listdir(OUT_DIR)):
    print(f"{name:<24} {os.path.getsize(os.path.join(OUT_DIR, name)) / 1024:>10,.1f} KB")

## B7. Sanity checks

In [ ]:
print("Shapes")
for name, obj in [("X_train", X_train), ("X_test", X_test),
                  ("y_train_reg", y_train_reg), ("y_test_reg", y_test_reg),
                  ("y_train_clf", y_train_clf), ("y_test_clf", y_test_clf)]:
    print(f"  {name:<12} {obj.shape}")
print(f"  test share: {len(X_test) / len(X):.3f}")

In [ ]:
dist = pd.concat([y_train_clf.value_counts(normalize=True), y_test_clf.value_counts(normalize=True)],
                 axis=1, keys=["train_%", "test_%"]).mul(100).round(2)
dist = dist.reindex(["Low", "Moderate", "High", "Severe"])
dist["diff"] = (dist["train_%"] - dist["test_%"]).round(2)
display(dist)

print(f"Mean burnout_score  train: {y_train_reg.mean():.4f} | test: {y_test_reg.mean():.4f}")
print(f"Std  burnout_score  train: {y_train_reg.std():.4f} | test: {y_test_reg.std():.4f}")

In [ ]:
forbidden = [ID_COL] + LEAKAGE_COLS + [TARGET_REG, TARGET_CLF]

# check the files on disk, not only the variables in memory
X_train_disk = pd.read_csv(f"{OUT_DIR}/X_train.csv")
X_test_disk = pd.read_csv(f"{OUT_DIR}/X_test.csv")

for name, d in [("X_train.csv", X_train_disk), ("X_test.csv", X_test_disk)]:
    left = [c for c in forbidden if c in d.columns]
    assert not left, f"{name} still contains {left}"
    assert d.columns.tolist() == FEATURES_FULL
    assert d.isna().sum().sum() == 0
assert not any(c in FEATURES_HR for c in PERSONAL_COLS)
assert len(X_train_disk) == len(pd.read_csv(f"{OUT_DIR}/y_train_reg.csv")) == len(pd.read_csv(f"{OUT_DIR}/y_train_clf.csv"))
assert len(X_test_disk) == len(pd.read_csv(f"{OUT_DIR}/y_test_reg.csv")) == len(pd.read_csv(f"{OUT_DIR}/y_test_clf.csv"))

print("OK – no ID, leakage or target columns in X, and X / y files have matching rows.")

**Versions.** The `.pkl` files only load safely with the same library versions, so the backend must install exactly these:

In [ ]:
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("joblib       :", joblib.__version__)
print("numpy        :", np.__version__)
print()
print(f"requirements.txt ->  pandas=={pd.__version__}  scikit-learn=={sklearn.__version__}  joblib=={joblib.__version__}  numpy=={np.__version__}")

Optional: download everything as one zip (Colab only).

In [ ]:
import shutil
shutil.make_archive("prepared", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("prepared.zip")
    files.download(CONSISTENT_PATH)
    files.download(CHANGES_PATH)
except ImportError:
    print("Not in Colab – files are in the working folder.")

## B8. How to use the files (for the two teammates)

Load the files, pick a preprocessor (`full` or `hr`) and put it in a `Pipeline` together with your model. The pipeline does imputing, scaling and encoding by itself, and when you call `.fit` it learns them from the training data only.

The models below are only placeholders and the `.fit` lines are commented out on purpose (this notebook doesn't train anything).

In [ ]:
import json, joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression   # placeholders, use any model

P = "prepared"

X_train = pd.read_csv(f"{P}/X_train.csv")
X_test = pd.read_csv(f"{P}/X_test.csv")
y_train_reg = pd.read_csv(f"{P}/y_train_reg.csv")["burnout_score"]
y_test_reg = pd.read_csv(f"{P}/y_test_reg.csv")["burnout_score"]
y_train_clf = pd.read_csv(f"{P}/y_train_clf.csv")["burnout_level"]
y_test_clf = pd.read_csv(f"{P}/y_test_clf.csv")["burnout_level"]

with open(f"{P}/feature_lists.json") as f:
    feature_lists = json.load(f)

# choose ONE: "full" or "hr"  (the X files are the same for both)
preprocessor = joblib.load(f"{P}/preprocessor_full.pkl")
# preprocessor = joblib.load(f"{P}/preprocessor_hr.pkl")

# --- Regression (burnout_score) ---
reg_pipeline = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
# reg_pipeline.fit(X_train, y_train_reg)
# y_pred = reg_pipeline.predict(X_test)

# --- Classification (burnout_level) ---
clf_pipeline = Pipeline([("prep", joblib.load(f"{P}/preprocessor_full.pkl")),
                         ("model", LogisticRegression(max_iter=1000))])
# clf_pipeline.fit(X_train, y_train_clf)
# y_pred = clf_pipeline.predict(X_test)

# To save a trained model for the backend (one file = preprocessing + model):
# joblib.dump(reg_pipeline, "burnout_regression_pipeline.pkl")

reg_pipeline

Notes for the teammates:

- Don't touch `X_test` / `y_test_*` until the very end. For tuning use cross-validation on the training data (`cross_val_score(pipeline, X_train, y_train, cv=5)`).
- Always pass the **whole** pipeline to cross-validation, not pre-transformed data, so each fold fits its own scaler.
- The class order for reports and confusion matrices is Low < Moderate < High < Severe (also saved in `feature_lists.json`).

---
# Final summary (for the report)

*The numbers below come from running this notebook on `mental_health_burnout_tech_2026.csv`. If the file changes, re-run and update them from the outputs of A2, A3, A5 and B7.*

**Dataset.** We used a synthetic dataset of 100,000 tech employees with 36 columns, no missing values and no duplicates. Because it is synthetic and some columns were generated independently of each other, it contained combinations that cannot exist in real life. **The data is not real, and we corrected it only for logical consistency. No values were changed to improve model performance and the target variables were never modified.**

**Gender filter.** We kept Male, Female and Prefer not to say. This removed 3,994 rows and left 96,006.

**What was wrong.** Out of the 96,006 rows:

| Rule | Problem | Rows breaking it |
|---|---|---|
| 1 | `years_experience > age - 18` (started working before 18) | 118 (0.12%) |
| 2 | `years_at_company > years_experience` (longer at the company than the whole career) | 17,374 (18.10%) |
| 3 | `seniority_level` too high for the experience (minimum years: Junior 0, Mid 2, Senior 5, Lead 6, Manager 6, Principal 10) | 19,031 (19.82%) |
| | At least one rule | 25,127 (26.17%) |

**Seniority thresholds.** There is no scientific standard for the minimum years per level, since every company defines its own levels. We based ours on published engineering career ladders and on real job postings from tech companies such as Asana (Senior 5+ years, Tech Lead 6+, Senior Staff 10+). Lead was set to 6 to match the Tech Lead requirement. These thresholds are the team's assumptions.

**How we fixed it (in this order).**

1. Rule 1: capped `years_experience` at `age - 18` (118 rows).
2. Rule 2: if the employee was old enough, we raised `years_experience` to `ceil(years_at_company)` (13,856 rows). Otherwise we capped `years_at_company` at `years_experience`, minimum 0.1 (3,520 rows). For 1,522 of those rows experience was 0, so it was set to 1 to keep the rule true.
3. Rule 3: we chose the smallest change. If the employee was old enough, we raised `years_experience` to the minimum for their level and kept the title (9,253 rows). Only if they were too young, we lowered `seniority_level` to the highest level their experience allows (7,187 rows). Together that is 16,440 rows, fewer than detected because Rule 2 had already raised the experience of some employees.

In total **25,127 rows (26.17%)** had at least one value changed, with 35,456 individual changes, all recorded in `changes_log.csv`. Only `years_experience`, `years_at_company` and `seniority_level` were touched. After the fixes all three checks return 0.

**Effect of the fixes.** The `burnout_level` distribution is identical before and after (Low 25.89%, Moderate 26.31%, High 19.26%, Severe 28.54%). The correlation of every numeric feature with `burnout_score` changed by at most 0.007 (our limit was 0.02). The one visible side effect is on `seniority_level`: in the original data Lead and Manager had much higher burnout than the other levels, so lowering a title moves that burnout to a lower level. The share of `burnout_score` variance explained by seniority alone went from 0.166 to 0.135. Lowering every impossible title instead of raising experience first would have dropped it to about 0.102, which is why we fixed experience first. Another side effect is that no employee has 0 years of experience after the fixes (8,669 did before), and mean experience rose from 9.34 to 9.85 years. We report both as limitations.

**Report-only checks.** IQR outliers were counted but not removed, because all values are inside realistic ranges. Median salary by country and seniority looks plausible (it rises with seniority and the USA is highest), except for a hard minimum of 40,000 USD that about 77% of employees in India and 45% in Brazil sit on.

**Columns dropped.**

- `employee_id`: an identifier with no predictive meaning.
- `stress_score`, `phq9_score`, `phq9_category`, `gad7_score`, `gad7_category`, `seeks_mental_health_support`, `job_change_intention`: leakage. They are mental-health outcomes that are strongly correlated with burnout (for example `stress_score` has a correlation of 0.85 with `burnout_score`) and an HR system would not have them.
- `burnout_score` and `burnout_level` are the targets and are not in the features, because each one reveals the other.

**Features.** `FEATURES_FULL` has 26 features. `FEATURES_HR` has 22: the same list without `sleep_hours_per_night`, `exercise_days_per_week`, `therapy_access` and `uses_therapy`, which a real HR system would not collect.

**Split.** One 80/20 split (`random_state=42`), stratified on `burnout_level`: 76,804 training rows and 19,202 test rows. The same rows are used for the regression and the classification target, so both models are evaluated on the same employees. Mean `burnout_score` is 5.394 in train and 5.401 in test.

**Preprocessing.** Numeric features: median imputation and standard scaling. `company_size`: ordinal encoding (Startup < Small < Mid < Large < Enterprise). Other categorical features: most-frequent imputation and one-hot encoding. The preprocessor was fitted on the training set only and saved, and the X files were saved unscaled so preprocessing and model live in one scikit-learn `Pipeline`.

---
# Suggestions (not applied)

Everything above follows the specification, with one change that was agreed: Rule 3 raises experience first and lowers the title only when the employee is too young (the original plan only lowered titles, which changed about 16,400 titles and weakened the seniority → burnout pattern from 0.166 to 0.102 instead of 0.135).

The points below are not applied. They are things I would consider, with the reason.

**1. Rule 2: compare whole years.**
`years_experience` is a whole number and `years_at_company` has decimals. Someone with 3 years of experience and 3.4 years at the company is not really impossible, the experience is just rounded down. Checking `floor(years_at_company) > years_experience` flags 11,869 rows instead of 17,374 (the 5,505 difference are only fractions of a year). Rounding the column itself (up or down) is not a good idea: rounding down puts about 30,000 employees at 0 years in the company, and rounding up adds about half a year to everyone without fixing anything. Use `floor` only inside the check. A side effect of the current rule is that nobody has 0 years of experience any more (8,669 people had 0 before), because any tenure above 0 is rounded up to 1.

**2. Rule 2b: a softer cap.**
When the tenure is impossible for the age, the current rule cuts it down to the (often very small) experience: for those 3,520 rows the average tenure goes from 7.7 to 1.7 years. An alternative is to first raise experience to `age - 18` and then cap the tenure at that (average 5.3). It changes the data less and the experience-0 edge case disappears by itself.

**3. Rule 3 never promotes anyone.**
A Junior with 25 years of experience is still in the data. This is not impossible, so leaving it is fine, but it is worth one sentence in the report so nobody thinks seniority and experience are fully aligned now.

**4. Make the `company_size` encoder safe for the backend.**
`OneHotEncoder` ignores unknown categories, but `OrdinalEncoder` raises an error if the backend sends a spelling it has never seen (for example `"Startup"` without `(1-50)`). I would add `handle_unknown="use_encoded_value", unknown_value=-1`, or validate the value in the backend before predicting.

**5. Encoding choices depend on the model.**
The ordinal `company_size` code (0–4) is not scaled like the other numeric columns. For tree models that doesn't matter; for linear models, KNN or SVM I would either scale it or one-hot it. `seniority_level` has a natural order too, so an ordinal version could be tried for tree models. The 0/1 columns don't need scaling, but scaling them does no harm.

**6. Split.**
80/20 is a good choice with 96,000 rows, I would keep it. I would only add that tuning should be done with 5-fold cross-validation on the training set so the test set is used once, at the end.

**7. Experience thresholds.**
Lead was changed from 7 to 6 to match real job postings (Asana asks for 6+ years for a Tech Lead). The other numbers already match published career ladders and job postings (Senior 5+, Senior Staff / Principal 10+). The weakest number is Manager (6), because management is a separate track and requirements vary a lot (public leveling guides use anywhere from 4+ to 6+ years). In the report, state clearly that these thresholds are the team's assumptions and cite the sources.